In [13]:
import tkinter as tk
from tkinter import filedialog, messagebox, scrolledtext
import librosa
import numpy as np
import pandas as pd
import joblib
import os

In [14]:
# Load scaler, optimized RandomForest model and RFE selector
scaler = joblib.load("../models/feature_scaler.pkl")
model = joblib.load("../models/rf_optimized.pkl")
selector = joblib.load("../models/rfe_selector.pkl")

# All features before feature selection
all_feature_names = [
    'mfcc_1_mean','mfcc_1_std','mfcc_2_mean','mfcc_2_std','mfcc_3_mean','mfcc_3_std',
    'mfcc_4_mean','mfcc_4_std','mfcc_5_mean','mfcc_5_std','mfcc_6_mean','mfcc_6_std',
    'mfcc_7_mean','mfcc_7_std','mfcc_8_mean','mfcc_8_std','mfcc_9_mean','mfcc_9_std',
    'mfcc_10_mean','mfcc_10_std','mfcc_11_mean','mfcc_11_std','mfcc_12_mean','mfcc_12_std',
    'mfcc_13_mean','mfcc_13_std','mfcc_14_mean','mfcc_14_std','mfcc_15_mean','mfcc_15_std',
    'chroma_1_mean','chroma_2_mean','chroma_3_mean','chroma_4_mean','chroma_5_mean',
    'chroma_6_mean','chroma_7_mean','chroma_8_mean','chroma_9_mean','chroma_10_mean',
    'chroma_11_mean','chroma_12_mean','contrast_1_mean','contrast_2_mean','contrast_3_mean',
    'contrast_4_mean','contrast_5_mean','contrast_6_mean','contrast_7_mean',
    'spec_centroid_mean','spec_bandwidth_mean','spec_rolloff_mean',
    'zcr_mean','rms_mean','tempo'
]

In [15]:
# Feature extraction function
def extract_features(filepath):
    y, sr = librosa.load(filepath, sr=22050)
    features = {}

    # MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=15)
    for i in range(mfcc.shape[0]):
        features[f'mfcc_{i+1}_mean'] = np.mean(mfcc[i])
        features[f'mfcc_{i+1}_std'] = np.std(mfcc[i])

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(chroma.shape[0]):
        features[f'chroma_{i+1}_mean'] = np.mean(chroma[i])

    # Spectral contrast
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        features[f'contrast_{i+1}_mean'] = np.mean(contrast[i])

    # Other spectral features
    features['spec_centroid_mean'] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    features['spec_bandwidth_mean'] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    features['spec_rolloff_mean'] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    features['zcr_mean'] = np.mean(librosa.feature.zero_crossing_rate(y))
    features['rms_mean'] = np.mean(librosa.feature.rms(y=y))

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features['tempo'] = tempo

    return features

# Function to browse and select multiple files
def upload_files():
    global selected_files
    filenames = filedialog.askopenfilenames(filetypes=[("Audio Files", "*.wav")])
    if filenames:
        selected_files = list(filenames)
        file_label.config(text=f"{len(selected_files)} files selected")

# Function to classify multiple audio files
def classify_audio_multiple():
    global selected_files
    if not selected_files:
        messagebox.showerror("Error", "Please upload audio files first!")
        return

    results = []

    for file in selected_files:
        try:
            features_dict = extract_features(file)
            features_df = pd.DataFrame([features_dict], columns=all_feature_names)

            features_scaled = scaler.transform(features_df)
            features_scaled = pd.DataFrame(features_scaled, columns=all_feature_names)

            # Apply feature selection
            features_selected = selector.transform(features_scaled)

            # Predict
            prediction = model.predict(features_selected)[0]

            results.append((os.path.basename(file), prediction))

        except Exception as e:
            results.append((os.path.basename(file), f"Error: {e}"))

    # Show results in a new window
    result_window = tk.Toplevel(root)
    result_window.title("Classification Results")
    result_window.geometry("400x300")

    text_area = scrolledtext.ScrolledText(result_window, wrap=tk.WORD, font=('Helvetica', 12))
    text_area.pack(expand=True, fill='both', padx=10, pady=10)

    for filename, pred in results:
        text_area.insert(tk.END, f"{filename} -> {pred}\n")

    text_area.configure(state='disabled')

In [16]:
# GUI
selected_files = []

root = tk.Tk()
root.title("Musical Genre Classifier")
root.configure(padx=20, pady=20)
default_font = ('Helvetica', 12)

title_label = tk.Label(root, text="Musical Genre Classifier", font=("Helvetica", 16, "bold"))
title_label.pack(pady=10)

file_label = tk.Label(root, text="No file selected", font=default_font, fg="gray")
file_label.pack(pady=10)

upload_button = tk.Button(root, text="Upload Files", command=upload_files, font=default_font, width=15)
upload_button.pack(pady=5)

classify_button = tk.Button(root, text="Classify Audio", command=classify_audio_multiple, font=default_font, bg="blue", fg="white", width=15)
classify_button.pack(pady=20)

root.mainloop()
